## Project Overview and Introduction

### Purpose
This project aims to develop a **Credit Transfer Analysis System** that automates and streamlines the process of evaluating External University course credits for transfer to an Internal University. The primary goal is to provide a robust and efficient mechanism for **Accessing** and **Assessing** the equivalence of courses taken at other institutions, ensuring academic integrity while simplifying administrative overhead.

### Problem
Credit transfer analysis is inherently complex, involving several challenges:
*  **Complexity in Access**: Administrative staff and faculty have to wait for official transcripts from the external university, which sometimes take weeks.
*   **Syllabus Comparison:** Accurately comparing course syllabi from different universities to determine content and learning outcome equivalence.
*   **Grade Verification:** Ensuring that the student's performance in the external course meets the internal university's grade requirements for transfer.
*   **Manual Effort:** Traditional methods often involve significant manual review by faculty and administrative staff, leading to delays and potential for human error.

### Multi-Agent System Approach
A multi-agent system architecture is particularly well-suited for this problem due to its inherent benefits:
*   **Modularity:** Complex tasks are broken down into smaller, manageable sub-tasks, each handled by a specialized agent (e.g., query extraction, external course info retrieval, internal course info retrieval, syllabus comparison).
*   **Distributed Processing:** Different agents can work concurrently on their respective tasks, improving efficiency and reducing processing time.
*   **Specialization:** Each agent is designed with specific instructions and tools to excel at its designated function, leading to more accurate and reliable outcomes.
*   **Flexibility and Scalability:** The system can be easily extended by adding new agents or modifying existing ones to handle new requirements or integrate with additional data sources without disrupting the entire system.
*   **Integartion** The A2A approach makes it easy by having specialized Agents expose only a specific functionality of an organization and be easily accessable to external organization.
*   **Robustness:** Failure of one agent does not necessarily bring down the entire system, as other agents can continue their operations, which is an usual complexity of Enterprise systems.


Let's start by installing googel ADK

In [ ]:
!pip install google-adk
!pip install -q google-adk[a2a]

Let's import the neccesarry lbraries needed for this project.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.adk.agents import LlmAgent, Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models import Gemini
from google.genai import types
import google.generativeai as genai
from google.colab import userdata
import os
import subprocess
import requests
import time
import uuid
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner, InMemoryRunner
from google.adk.tools import AgentTool
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from google.adk import a2a
from google.adk.a2a.utils.agent_to_a2a import to_a2a

This will be our helper function to use when we encounter transient errors like rate limits or temporary service unavailability. Retry options automatically handle these failures by retrying the request with exponential backoff.

In [ ]:
retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)

In [ ]:
# The below code connects to google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import google.generativeai as genai # For using Google's Gemini LLM
from google.colab import userdata # For securely storing your API key
import os

# Get your Gemini API key from Colab secrets
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

# Set the API key as an environment variable to ensure it's picked up by internal clients
os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY

This project uses two datasets. **df_lus** will mimic Student Course information from an external university from which the Student intends to transfer courses from. This will be used by the External University system Agent. And **df_cus** dataset, which mimics Internal/current Student Course Information databse from the current university to which the credits should be transffered to. These dataframes are extracted from respective excel files using Pandas.

In [ ]:
df_lus = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/Kaggle Competitions/LUS_Student_crs_info.xlsx')
df_cus = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/Kaggle Competitions/CUS_Student_crs_info.xlsx')

print("✅ LUS_Student_crs_info.xlsx loaded into df_lus.")
print("✅ CUS_Student_crs_info.xlsx loaded into df_cus.")

print("df_lus head:")
print(df_lus.head())
print("\ndf_cus head:")
print(df_cus.head())

✅ LUS_Student_crs_info.xlsx loaded into df_lus.
✅ CUS_Student_crs_info.xlsx loaded into df_cus.
df_lus head:
  student_id         crs_major crs_num                    crs_title  \
0      S1001  Computer Science  CS-301         Intro to Programming   
1      S1001                AI  AI-625  Deep Learning Architectures   
2      S1001       Mathematics  MA-412             Complex Analysis   
3      S1001  Computer Science  CS-450            Operating Systems   
4      S1002      Data Science  DS-510         Statistical Modeling   

                                     crs_description  \
0  Basic concepts of structured programming and a...   
1  Study of advanced neural network models and th...   
2  Theory and application of functions of a compl...   
3  Principles, design, and implementation of oper...   
4  Core principles of inferential statistics and ...   

                                        crs_syllabus  num_of_credits  \
0  Design and implement algorithms using foundati...   

Now, we create our fist Agent **query_extraction_agent**, whose sole reponsibility is to take a user'natural laguage query and extract key information, here it's **student_num**, **internal_crs_num** and **external_crs_num**. We give a clear instruction that the output should be a Python dictionary string literal with repective keys. And we store it in the output_key "extracted_query_params".  If a value is not found in the query, the corresponding value in the dictionary will be None.  This is an LLM agent designed for precise entity extraction from free-form text, ensuring structured input for subsequent processing.

In [ ]:
query_extraction_agent = Agent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="query_extraction_agent",
    description="Agent solely responsible for extracting student_id, internal_crs_num, and external_crs_num from user queries.",
    instruction="""
    Your sole task is to extract 'student_id', 'internal_crs_num', and 'external_crs_num' from the user's query.
    You MUST output the extracted information as a Python dictionary string literal with the keys: 'student_id', 'internal_crs_num', and 'external_crs_num'.
    For example: {'student_id': 'S1001', 'internal_crs_num': 'CS-300', 'external_crs_num': 'CS-301'}.
    If any of these pieces of information are not found in the query, the corresponding value in the dictionary output MUST be 'None'.
    DO NOT include any JSON markdown formatting (e.g., ```json) or any other extra text in your output; return only the dictionary string literal.
    """,
    output_key="extracted_query_params",
)

print("✅ Query Extraction Agent created successfully!")
print("   Model: gemini-2.5-flash-lite")
print("   Output Key: extracted_query_params")

✅ Query Extraction Agent created successfully!
   Model: gemini-2.5-flash-lite
   Output Key: extracted_query_params


Let's quickly test this agent.

In [ ]:
runner = InMemoryRunner(agent=query_extraction_agent)
response = await runner.run_debug("Initialize Transfer credit for student_id 'S10002', internal_crs_num 'CS-300', and external_crs_num 'CS-301'")



 ### Created new session: debug_session_id

User > Initialize Transfer credit for student_id 'S10002', internal_crs_num 'CS-300', and external_crs_num 'CS-301'
query_extraction_agent > {'student_id': 'S10002', 'internal_crs_num': 'CS-300', 'external_crs_num': 'CS-301'}


We now create python function **get_internal_student_course_info** which is responsible for retrieving internal university course information from our internal university dataset **df_cus** DataFrame. It takes **student_id** and **crs_num** as string for input, uses df_cus dataframe and returns relevant records as a list of dictionaries containing course details, if a match is found. If no information is found or an error occurs, it returns an informative string message.
This function includes `print` statements prefixed with **[OBSERVABILITY]** to trace its execution flow, input parameters, and return values. These statements are crucial for debugging and monitoring the agent's behavior during development and runtime.

In [ ]:
from typing import Union, List, Dict # Add this import
def get_internal_student_course_info(student_id: str, crs_num: str) -> Union[str, List[Dict]]:
    """Get course information for a given student ID and course number.

    Args:
        student_id: The ID of the student (e.g., "S1002").
        crs_num: The course number (e.g., "CS-101").


    Returns:
        A list of dictionaries representing the student's course information,
        or an error/informational message string if no information is found
        or an error occurs.
    """
    print(f"[OBSERVABILITY] get_internal_student_course_info called with student_id={student_id}, crs_num={crs_num}, df_cus_shape={df_cus.shape}")
    try:
        # Ensure both the input student_id and crs_num and the DataFrame columns are strings for comparison
        df_cus['student_id'] = df_cus['student_id'].astype(str)
        df_cus['crs_num'] = df_cus['crs_num'].astype(str)
        student_id_str = str(student_id)
        crs_num_str = str(crs_num)

        # Filter by both student_id and crs_num
        matching_records = df_cus[(df_cus['student_id'] == student_id_str) & (df_cus['crs_num'] == crs_num_str)]

        if not matching_records.empty:
             result = matching_records.to_dict(orient='records')
             print(f"[OBSERVABILITY] get_internal_student_course_info returned result: {result}")
             return result
        else:
            message = f"No course info found for Student {student_id_str} and Course {crs_num_str}"
            print(f"[OBSERVABILITY] get_internal_student_course_info returned message: {message}")
            return message
    except Exception as e:
        message = f"An error occurred: {e}"
        print(f"[OBSERVABILITY] get_internal_student_course_info returned error: {message}")
        return message

Let's give a quick test

In [ ]:
get_internal_student_course_info('S1001', 'CS-300')

[OBSERVABILITY] get_internal_student_course_info called with student_id=S1001, crs_num=CS-300, df_cus_shape=(100, 9)
[OBSERVABILITY] get_internal_student_course_info returned result: [{'student_id': 'S1001', 'crs_major': 'Computer Science', 'crs_num': 'CS-300', 'crs_title': 'Foundations of Programming', 'crs_description': 'Basic concepts of structured programming and algorithm design.', 'crs_syllabus': 'Design and implement algorithms using foundational control structures. Explain basic object-oriented concepts like encapsulation and inheritance. Debug and test structured programming code effectively. Utilize and manipulate basic data structures like arrays and linked lists.', 'num_of_credits': 4, 'num_of_credits_earned': 4, 'final_grade': 'A'}]


[{'student_id': 'S1001',
  'crs_major': 'Computer Science',
  'crs_num': 'CS-300',
  'crs_title': 'Foundations of Programming',
  'crs_description': 'Basic concepts of structured programming and algorithm design.',
  'crs_syllabus': 'Design and implement algorithms using foundational control structures. Explain basic object-oriented concepts like encapsulation and inheritance. Debug and test structured programming code effectively. Utilize and manipulate basic data structures like arrays and linked lists.',
  'num_of_credits': 4,
  'num_of_credits_earned': 4,
  'final_grade': 'A'}]

Next, we create a python function compare_syllabi which quantify and qualitatively assesses the similarity between two given course syllabi. It utilizes TfidfVectorizer and cosine_similarity to calculate the similarity score. It takes two strings syllabus1 and syllabus2 and returns a tuple containing a numerical similarity score (float) and a qualitative assessment (string, e.g., 'Excellent Match', 'No Match'). The function incorporates `print` statements, prefixed with **[OBSERVABILITY]**, to enhance debugging and monitoring capabilities. These statements trace:
1. The function's invocation, including the lengths of the input syllabi.
2. The final computed similarity score and its corresponding qualitative assessment. This allows developers and system administrators to understand the function's execution flow and outcomes during runtime.

In [ ]:
def compare_syllabi(syllabus1: str, syllabus2: str):
    """Calculates the similarity between two syllabi and provides a qualitative assessment."""
    print(f"[OBSERVABILITY] compare_syllabi called with syllabus1 (length {len(syllabus1)}) and syllabus2 (length {len(syllabus2)})")
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform([syllabus1, syllabus2])
    similarity_score = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]

    if similarity_score >= 0.8:
        assessment = 'Excellent Match'
    elif similarity_score >= 0.6:
        assessment = 'Good Match'
    elif similarity_score >= 0.4:
        assessment = 'Partial Match'
    elif similarity_score >= 0.2:
        assessment = 'Poor Match'
    else:
        assessment = 'No Match'

    print(f"[OBSERVABILITY] compare_syllabi returned score={similarity_score:.2f}, assessment='{assessment}'")
    return similarity_score, assessment


From here on we will be wrting our code for the **external_student_course_info_agent**  that provides external course information. It serves as the agent that will be exposed via **A2A** communication. It takes a request like a request string for course information (e.g., "Get course information for student ID 'S1005' and external_crs_num 'CS-502'"). and returns Course information as a list of dictionaries, or an informational string .It uses the **get_student_course_info** tool internally and is intended to be run remotely.





**I will be submitting this external_student_course_info_agent code as a separate file with Evaluation that can be tested separately. In this file, I will have all the data and code for External University Student Course Info added as part of the external_student_course_info_agent_code, that can be imported by uvicorn as a file.**

We will write the **external_student_course_info_agent**(working code presented as a separate file) definition and deploy it as a separate, remote service. This is a critical step for demonstrating **Agent-to-Agent (A2A) communication** in this project, as it simulates the agent running independently on a different server.
1. First, we **save df_lus to CSV**: The external university's course data (df_lus) is first saved to a temporary CSV file (/tmp/df_lus.csv). This is done so the remote agent, when launched, can load this data independently.
2. Next, we **Generate Agent Code**: The Python code defining the ***external_student_course_info_agent*** and its tools ***query_extraction_agent***, ***async def get_extracted_params***, and ***get_student_course_info*** is encapsulated in a string ***external_student_course_info_agent_code***.
3. We then write this to a **File**: This code string is then written to a Python file (/tmp/external_student_course_info_server.py)
4. **Start Uvicorn Server**: A uvicorn server is launched in a background process, serving the FastAPI application generated from the external_student_course_info_agent on http://localhost:8001. This makes the agent accessible remotely via HTTP.
5. and finally, we **check the Server Readiness**: The code then polls the remote agent's .well-known/agent-card.json endpoint to ensure the server is up and responsive before proceeding.
This setup allows the root_agent(orchestrater) to interact with external_student_course_info_agent through the remote_external_student_course_info_agent proxy, as described in the next sections.

Here is a description of the tools and agent that are part of this process:

**query_extraction_agent**: query_extraction_agent, whose sole reponsibility is to take a user'natural laguage query and extract key information, here it's student_num, and crs_num. We give a clear instruction that the output should be a Python dictionary string literal with repective keys. And we store it in the output_key "extracted_query_params". If a value is not found in the query, the corresponding value in the dictionary will be None. This is an LLM agent designed for precise entity extraction from free-form text, ensuring structured input for subsequent processing.

**async def get_extracted_params**:  Asynchronous Python helper function that explicitly runs the query_extraction_agent using its own runner and extracts the parameters. This was developed during the evaluation to simplify the tool call for the LLM agent.

**get_student_course_info**:Extracts information from our external univesity database.This is an internal Python function in the external university system responsible for retrieving course information from the df_lus DataFrame taking student_num and crs_num as inputs and outputs list of dictionaries containing course details, or a string message if no information is found or an error occurs. This function also has print statements prefixed with [OBSERVABILITY] to trace its execution flow, input parameters, and return values.

**external_student_course_info_agent_code** The main orchestrator of the external university system that provides external course information when called upon. It serves as the agent that will be exposed via A2A communication. It takes a request like a request string for course information (e.g., "Get course information for student ID 'S1005' and external_crs_num 'CS-502'"). and returns Course information as a list of dictionaries, or an informational string .It uses the get_student_course_info tool internally and is intended to be run remotely.


**Rerun cells from here until root_agent if encontered any issues with testing.**

In [ ]:
import subprocess
import requests
import time
import pandas as pd # Import pandas here for saving df_lus

# Save df_lus to a temporary CSV file
df_lus.to_csv('/tmp/df_lus.csv', index=False)
print("📝 df_lus saved to /tmp/df_lus.csv")

# First, let's save the External Student Course Info agent to a file that uvicorn can import
external_student_course_info_agent_code = '''
import os
import pandas as pd # For data manipulation and analysis
import numpy as np # For numerical computing
import matplotlib.pyplot as plt # For data visualization
from google.adk.agents import LlmAgent, Agent
from google.adk import a2a
from google.adk.a2a.utils.agent_to_a2a import to_a2a
from google.adk.models.google_llm import Gemini

from google.genai import types
from typing import Union, List, Dict # ADDED THIS IMPORT
import json # ADDED THIS IMPORT

import ast
from google.adk.tools import FunctionTool # Import FunctionTool
from google.adk.runners import InMemoryRunner # ADDED THIS IMPORT

# Load df_lus from the temporary CSV file
df_lus = pd.read_csv('/tmp/df_lus.csv')

retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)

#create query extraction agent
query_extraction_agent = Agent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="query_extraction_agent",
    description="Agent solely responsible for extracting student_id and external_crs_num from user queries.",
    instruction="""
    Your sole task is to extract 'student_id', 'external_crs_num' from the user's query.
    You MUST output the extracted information as a Python dictionary string literal with the keys: 'student_id', and 'external_crs_num'.
    For example: {'student_id': 'S1001', 'external_crs_num': 'CS-301'}.
    If any of these pieces of information are not found in the query, the corresponding value in the dictionary output MUST be 'None'.
    DO NOT include any JSON markdown formatting (e.g., ```json) or any other extra text in your output; return only the dictionary string literal.
    """,
    output_key="extracted_query_params",
)

print("✅ Query Extraction Agent created successfully!")
print("   Model: gemini-2.5-flash-lite")
print("   Output Key: extracted_query_params")


#create external student course info agent
def get_student_course_info(student_id: str, crs_num: str) -> Union[str, List[Dict]]:
    """Gets course information for a given student ID and course number from df_lus dataframe.

    This tool retrievs complete course information for a student and course number
    This information will be compared with an external university
    course for a potential credit transfer.

    Args:
        student_id: The ID of the student (e.g., "S1002").
        crs_num: The course number (e.g., "CS-101").


    Returns:
        A list of dictionaries representing the student's course information,
        or an error/informational message string if no information is found
        or an error occurs.
    """
    print(f"[OBSERVABILITY] get_student_course_info for external student called with student_id={student_id}, crs_num={crs_num}, df_lus_shape={df_lus.shape}")
    try:
        # Ensure both the input student_id and crs_num and the DataFrame columns are strings for comparison
        df_lus['student_id'] = df_lus['student_id'].astype(str)
        df_lus['crs_num'] = df_lus['crs_num'].astype(str)
        student_id_str = str(student_id)
        crs_num_str = str(crs_num)

        # Filter by both student_id and crs_num
        matching_records = df_lus[(df_lus['student_id'] == student_id_str) & (df_lus['crs_num'] == crs_num_str)]

        if not matching_records.empty:
             result = matching_records.to_dict(orient='records')
             print(f"[OBSERVABILITY] get_student_course_info for external student returned result: {result}")
             return result
        else:
            message = f"No course info found for Student {student_id_str} and Course {crs_num_str}"
            print(f"[OBSERVABILITY] get_student_course_info for external student returned message: {message}")
            return message
    except Exception as e:
        message = f"An error occurred: {e}"
        print(f"[OBSERVABILITY] get_student_course_info for external student returned error: {message}")
        return message



# Define the asynchronous Python helper function get_extracted_params
async def get_extracted_params(query: str) -> dict:
    """Helper function to run query_extraction_agent and return extracted parameters."""
    print(f"[OBSERVABILITY] get_extracted_params called for query: {query}")
    local_query_runner = InMemoryRunner(agent=query_extraction_agent)
    response_events = await local_query_runner.run_debug(query)

    extracted_params_output_str = None
    for event in response_events:
        if hasattr(event, 'content') and event.content and \
           hasattr(event.content, 'role') and event.content.role == 'model':
            if hasattr(event.content, 'parts') and event.content.parts is not None:
                text_parts = [part.text for part in event.content.parts if hasattr(part, 'text') and part.text is not None]
                if text_parts:
                    extracted_params_output_str = " ".join(text_parts).strip()
                    break

    extracted_params = {'student_id': None, 'external_crs_num': None}
    if extracted_params_output_str:
        try:
            extracted_params = ast.literal_eval(extracted_params_output_str)
        except (ValueError, SyntaxError) as e:
            print(f"[ERROR] Error evaluating query_extraction_agent output: {extracted_params_output_str} - {e}")
    print(f"[OBSERVABILITY] get_extracted_params returned: {extracted_params}")
    return extracted_params


# Redefine the external_student_course_info_agent with refined instructions
# (Instruction from previous step, ensuring error message has no parentheses and JSON is pure array)
external_student_course_info_agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="external_student_course_info_agent",
    description="External University's Student Information Agent that provides Course information of Students ",
    instruction="""
    You are a Student Course Information specialist of an External University.
    When asked about Course Information for a student, you must follow these steps:
    1. Call the `get_extracted_params` tool with `query=user_query`. This output will be accessible as `extracted_query_params`.
    2. IF `extracted_query_params['student_id']` is None OR `extracted_query_params['external_crs_num']` is None:
       THEN YOUR FINAL AND ONLY RESPONSE MUST BE 'I cannot proceed because student ID or external course number is missing. Please provide all necessary details.' and STOP. DO NOT vary this message.
    3. ELSE (both parameters are present):
       Call the `get_student_course_info` tool with `student_id=extracted_query_params['student_id']` and `crs_num=extracted_query_params['external_crs_num']`. Let the result be `external_course_info_raw`.
       a. IF `external_course_info_raw` is a string (indicating no course info or an error message):
          THEN YOUR FINAL RESPONSE MUST BE the content of `external_course_info_raw` and STOP.
       b. ELSE IF `external_course_info_raw` is an empty list:
          THEN YOUR FINAL RESPONSE MUST BE 'No external course information found for the provided details.' and STOP.
       c. ELSE (it is a non-empty list of dictionaries):
          THEN YOUR FINAL AND ONLY RESPONSE MUST BE the JSON string representation of `external_course_info_raw`. Ensure it is a valid JSON array of objects, starting with `[` and ending with `]`, and contains no other text or markdown like ```json. CRITICALLY, DO NOT wrap it in any additional dictionary like {"result": [...]}. YOUR FINAL AND ONLY RESPONSE MUST BE THIS PURE JSON ARRAY STRING.

    Be professional and helpful in all your responses.
    """,
    tools=[FunctionTool(get_extracted_params), get_student_course_info],
)

print("✅ external_student_course_info_agent redefined with refined instructions.")

# Create the A2A app
app = to_a2a(external_student_course_info_agent, port=8001)
'''
# Terminate existing server process if it's running
if "external_student_course_info_server_process" in globals() and globals()["external_student_course_info_server_process"].poll() is None:
    print("Stopping existing External Student Course Info Agent server...")
    globals()["external_student_course_info_server_process"].terminate()
    globals()["external_student_course_info_server_process"].wait() # Wait for termination
    print("Existing server stopped.")

# Write the Student Course Info agent to a temporary file
with open("/tmp/external_student_course_info_server.py", "w") as f:
    f.write(external_student_course_info_agent_code)

print("📝 Student Course Info agent code saved to /tmp/external_student_course_info_server.py")

# Start uvicorn server in background
# Note: We redirect output to avoid cluttering the notebook
server_process = subprocess.Popen(
    [
        "uvicorn",
        "external_student_course_info_server:app",  # Module:app format
        "--host",
        "localhost",
        "--port",
        "8001",
    ],
    cwd="/tmp",  # Run from /tmp where the file is
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env={**os.environ},  # Pass environment variables (including GOOGLE_API_KEY)
)

print("🚀 Starting External Student Course Info Agent server...")
print("   Waiting for server to be ready...")

# Wait for server to start (poll until it responds)
max_attempts = 30
for attempt in range(max_attempts):
    try:
        response = requests.get(
            "http://localhost:8001/.well-known/agent-card.json", timeout=1
        )
        if response.status_code == 200:
            print(f"\n✅ External Student Course Info Agent server is running!")
            print(f"   Server URL: http://localhost:8001")
            print(f"   Agent card: http://localhost:8001/.well-known/agent-card.json")
            break
    except requests.exceptions.RequestException:
        time.sleep(5)
        print(".", end="", flush=True)
else:
    print("\n⚠️  Server may not be ready yet. Check manually if needed.")

# Store the process so we can stop it later
globals()["external_student_course_info_server_process"] = server_process

📝 df_lus saved to /tmp/df_lus.csv
Stopping existing External Student Course Info Agent server...
Existing server stopped.
📝 Student Course Info agent code saved to /tmp/external_student_course_info_server.py
🚀 Starting External Student Course Info Agent server...
   Waiting for server to be ready...
....
✅ External Student Course Info Agent server is running!
   Server URL: http://localhost:8001
   Agent card: http://localhost:8001/.well-known/agent-card.json


We can now Fetch and print the agent card from the running server

In [ ]:
import time
import json
# Fetch and print the agent card from the running server
try:
    response = requests.get(
        "http://localhost:8001/.well-known/agent-card.json", timeout=5
    )

    if response.status_code == 200:
        agent_card = response.json()
        print("\n📋 External Student Course Info Agent Card (External):")
        print(json.dumps(agent_card, indent=2))

        print("\n✨ Key Information:")
        print(f"   Name: {agent_card.get('name')}")
        print(f"   Description: {agent_card.get('description')}")
        print(f"   URL: {agent_card.get('url')}")
        print(f"   Skills: {len(agent_card.get('skills', []))} capabilities exposed")
    else:
        print(f"\n❌ Failed to fetch agent card: {response.status_code}")

except requests.exceptions.RequestException as e:
    print(f"\n❌ Error fetching agent card: {e}")
    print("   Make sure the External Student Course Info Agent server is running.")


📋 External Student Course Info Agent Card (External):
{
  "capabilities": {},
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain"
  ],
  "description": "External University's Student Information Agent that provides Course information of Students ",
  "name": "external_student_course_info_agent",
  "preferredTransport": "JSONRPC",
  "protocolVersion": "0.3.0",
  "skills": [
    {
      "description": "External University's Student Information Agent that provides Course information of Students  \n    I am a Student Course Information specialist of an External University.\n    When asked about Course Information for a student, I must follow these steps:\n    1. Call the `get_extracted_params` tool with `query=user_query`. This output will be accessible as `extracted_query_params`.\n    2. IF `extracted_query_params['student_id']` is None OR `extracted_query_params['external_crs_num']` is None:\n       THEN my FINAL AND ONLY RESPONSE MUST BE 'I cannot

Next we create the **RemoteA2aAgent** Proxy.
This step establishes the connection between our CreditsAnalysisCoordinator (**root agent**) and the remotely deployed **external_student_course_info_agent**. It does so by creating a RemoteA2aAgent instance, **remote_external_student_course_info_agent**.
###The purpose:
1. **Client-Side Proxy**: The remote_external_student_course_info_agent acts as a client-side proxy within the main notebook environment. It allows the CreditsCoordinator to interact with the external agent as if it were a local sub-agent.
2. **Abstracts A2A Communication**: This proxy handles the complexities of Agent-to-Agent (A2A) communication, including network calls, request/response serialization (converting Python objects to data transferable over the network) and deserialization (converting received network data back into Python objects).
3. **Configuration**: It is configured with the name of the remote agent and the URL to its agent_card.json endpoint (e.g., http://localhost:8001/.well-known/agent-card.json). This agent card provides the necessary metadata for the proxy to understand the remote agent's capabilities.
4. **Benefit**: By using this proxy, the CreditsAnalysisCoordinator can call the external agent's tools (like get_external_student_course_info) using a familiar Python function-call syntax, without needing to worry about the underlying HTTP requests or data formatting.

In [ ]:
# Create a RemoteA2aAgent that connects to our External Student Course Info Agent
# This acts as a client-side proxy - the Registrar Support Agent can use it like a local agent
AGENT_CARD_WELL_KNOWN_PATH = '/.well-known/agent-card.json'
remote_external_student_course_info_agent = RemoteA2aAgent(
    name="external_student_course_info_agent",
    description="Remote External Student course Info agent from external university that provides course information for a given student at their university.",
    agent_card=f"http://localhost:8001{AGENT_CARD_WELL_KNOWN_PATH}",
)

print("✅ External Student Course Info proxy created!")
print(f"   Connected to: http://localhost:8001")
print(f"   Agent card: http://localhost:8001{AGENT_CARD_WELL_KNOWN_PATH}")
print("   The Registrar Support Agent can now use this like a local sub-agent!")

✅ External Student Course Info proxy created!
   Connected to: http://localhost:8001
   Agent card: http://localhost:8001/.well-known/agent-card.json
   The Registrar Support Agent can now use this like a local sub-agent!


/tmp/ipython-input-3042181325.py:4: UserWarning: [EXPERIMENTAL] RemoteA2aAgent: ADK Implementation for A2A support (A2aAgentExecutor, RemoteA2aAgent and corresponding supporting components etc.) is in experimental mode and is subjected to breaking changes. A2A protocol and SDK arethemselves not experimental. Once it's stable enough the experimental mode will be removed. Your feedback is welcome.
  remote_external_student_course_info_agent = RemoteA2aAgent(


Now, we create our root_agent **CreditsAnalysisCoordinator**, which is the **main orchestrator agent** responsible for the entire credit transfer analysis workflow. It takes a natural language query from the user specifying the student ID and both internal and external course number and  It orchestrates a sequential workflow, including:
1. **Parameter extraction** using query_extraction_agent.
2. **External course information retrieval and grade checking** using remote_external_student_course_info_agent.
3. **Internal course information retrieval** using get_internal_student_course_info.
4. **Syllabus comparison** using compare_syllabi.
5. Finally, **synthesizing all information** to produce the credit transfer analysis.

In [ ]:
root_agent = LlmAgent(model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
                      name = "CreditsAnalysisCoordinator",
                      description = "An orchestrator agent for Credits Analysis coordinator",
                      instruction ="""You are a Transfer Course Credits coordinator. Your goal is to answer the user's query by orchestrating a workflow.
* Use query_extraction_agent() tool to extract `student_id`, `internal_crs_num`, and `external_crs_num` from the user's query. This output will be accessible as `extracted_query_params`.
* If `extracted_query_params['student_id']` is None or `extracted_query_params['external_crs_num']` is None or `extracted_query_params['internal_crs_num']` is None, then output 'I cannot proceed because crucial information (student ID, internal course number, or external course number) is missing. Please provide all necessary details.' and stop.
* Output 'Parameters extracted successfully.'

* Construct a request string for the external_student_course_info_agent tool: "Get course information for student ID '{extracted_query_params['student_id']}' and course number '{extracted_query_params['external_crs_num']}'."
* Call external_student_course_info_agent(request=this_constructed_string). Let the result be `external_course_info`.


* Store `external_course_info[0]['crs_syllabus']` as `external_syllabus`.
* Check if `external_course_info[0]['final_grade']` is 'A' or 'B'. If not, output 'The external course cannot be transferred due to grade requirements (only A or B are accepted).' and stop the process.
* Output 'External course information retrieved and grade is acceptable.'

* Call get_internal_student_course_info(student_id=extracted_query_params['student_id'], crs_num=extracted_query_params['internal_crs_num']). Let the result be `internal_course_info_raw`.
* If `internal_course_info_raw` is a string (meaning no course info or error), output `internal_course_info_raw` as the response and stop the process.
* If `internal_course_info_raw` is an empty list, output 'No internal course information found for the provided details.' and stop the process.
* Assume `internal_course_info_raw` is now a non-empty list of dictionaries. Let `internal_course_info = internal_course_info_raw`.

* Store `internal_course_info[0]['crs_syllabus']` as `internal_syllabus`.
* Output 'Internal course information retrieved.'

* Call compare_syllabi(syllabus1=internal_syllabus, syllabus2=external_syllabus). Let the result be `syllabus_comparison`.

* Output the following summary, ensuring all values are correctly formatted and available:
  'Credit Transfer Analysis Results:\n'
  'External Course:\n'
  '  Student ID: {extracted_query_params['student_id']}\n'
  '  Course Number: {extracted_query_params['external_crs_num']}\n'
  '  Title: {external_course_info[0]['crs_title']}\n'
  '  Grade: {external_course_info[0]['final_grade']}\n'
  'Internal Course:\n'
  '  Student ID: {extracted_query_params['student_id']}\n'
  '  Course Number: {extracted_query_params['internal_crs_num']}\n'
  '  Title: {internal_course_info[0]['crs_title']}\n'
  'Syllabus Comparison:\n'
  '  Similarity Score: {syllabus_comparison[0]:.2f}\n'
  '  Assessment: {syllabus_comparison[1]}'""",
                      tools = [AgentTool(query_extraction_agent), AgentTool(remote_external_student_course_info_agent), get_internal_student_course_info, compare_syllabi],

)

Before we try testing, I do want to mention here that the biggest limitation here is the A2a with uvicorn. There is a big deal of server becoming unavailable. This is the main reason of not being able to add Evaluation directly to this project as on multiple runs the server becomes unavailable. So, if you run into this issue while testing this, please rerun the cell from where we save the external_student_crs_info code until the root_agent and run the tests.

Now let's try to test some senarios. The first one is where the external course **grade > C** and **perfect syllabus match** for both courses. External and Internal Course Syllabus **similary score is 1 perfect match.**

In [ ]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug("Initialize Transfer credit for student_id 'S1001', internal_crs_num 'CS-300', and external_crs_num 'CS-301'")



 ### Created new session: debug_session_id

User > Initialize Transfer credit for student_id 'S1001', internal_crs_num 'CS-300', and external_crs_num 'CS-301'
CreditsAnalysisCoordinator > Parameters extracted successfully.



/usr/local/lib/python3.12/dist-packages/google/adk/agents/remote_a2a_agent.py:389: UserWarning: [EXPERIMENTAL] convert_genai_part_to_a2a_part: ADK Implementation for A2A support (A2aAgentExecutor, RemoteA2aAgent and corresponding supporting components etc.) is in experimental mode and is subjected to breaking changes. A2A protocol and SDK arethemselves not experimental. Once it's stable enough the experimental mode will be removed. Your feedback is welcome.
  converted_parts = self._genai_part_converter(part)
/usr/local/lib/python3.12/dist-packages/google/adk/a2a/converters/event_converter.py:239: UserWarning: [EXPERIMENTAL] convert_a2a_message_to_event: ADK Implementation for A2A support (A2aAgentExecutor, RemoteA2aAgent and corresponding supporting components etc.) is in experimental mode and is subjected to breaking changes. A2A protocol and SDK arethemselves not experimental. Once it's stable enough the experimental mode will be removed. Your feedback is welcome.
  return convert_a

CreditsAnalysisCoordinator > External course information retrieved and grade is acceptable.

[OBSERVABILITY] get_internal_student_course_info called with student_id=S1001, crs_num=CS-300, df_cus_shape=(100, 9)
[OBSERVABILITY] get_internal_student_course_info returned result: [{'student_id': 'S1001', 'crs_major': 'Computer Science', 'crs_num': 'CS-300', 'crs_title': 'Foundations of Programming', 'crs_description': 'Basic concepts of structured programming and algorithm design.', 'crs_syllabus': 'Design and implement algorithms using foundational control structures. Explain basic object-oriented concepts like encapsulation and inheritance. Debug and test structured programming code effectively. Utilize and manipulate basic data structures like arrays and linked lists.', 'num_of_credits': 4, 'num_of_credits_earned': 4, 'final_grade': 'A'}]
CreditsAnalysisCoordinator > Internal course information retrieved.

[OBSERVABILITY] compare_syllabi called with syllabus1 (length 276) and syllabus2 (

Result when External grade **> C** but **poor couse syllabus match** for both courses. The similarity score 0.11. Poor Match.

In [ ]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug("Initialize Transfer credit for student_id 'S1010', internal_crs_num 'MA-373', and external_crs_num 'MA-705'")


 ### Created new session: debug_session_id

User > Initialize Transfer credit for student_id 'S1010', internal_crs_num 'MA-373', and external_crs_num 'MA-705'
CreditsAnalysisCoordinator > Parameters extracted successfully.

CreditsAnalysisCoordinator > External course information retrieved and grade is acceptable.

[OBSERVABILITY] get_internal_student_course_info called with student_id=S1010, crs_num=MA-373, df_cus_shape=(100, 9)
[OBSERVABILITY] get_internal_student_course_info returned result: [{'student_id': 'S1010', 'crs_major': 'Mathematics', 'crs_num': 'MA-373', 'crs_title': 'Differential Equations', 'crs_description': 'Solving ordinary and partial differential equations.', 'crs_syllabus': 'Solve first and second-order ODEs. Analyze stability of dynamical systems. Use Laplace transforms to solve IVPs. Introduce basic concepts of PDEs.', 'num_of_credits': 4, 'num_of_credits_earned': 4, 'final_grade': 'A'}]
CreditsAnalysisCoordinator > Internal course information retrieved.

[OBSER

Result when External **grade < C** so course doesn not satisfy the grade requirement. Notice here, the the process stopped, without calling the internal course tool.

In [ ]:
response = await runner.run_debug("Initialize Transfer credit for student_id 'S1005', internal_crs_num 'PH-348', and external_crs_num 'PH-350'")


 ### Continue session: debug_session_id

User > Initialize Transfer credit for student_id 'S1005', internal_crs_num 'PH-348', and external_crs_num 'PH-350'
CreditsAnalysisCoordinator > Parameters extracted successfully.



/usr/local/lib/python3.12/dist-packages/google/adk/agents/remote_a2a_agent.py:389: UserWarning: [EXPERIMENTAL] convert_genai_part_to_a2a_part: ADK Implementation for A2A support (A2aAgentExecutor, RemoteA2aAgent and corresponding supporting components etc.) is in experimental mode and is subjected to breaking changes. A2A protocol and SDK arethemselves not experimental. Once it's stable enough the experimental mode will be removed. Your feedback is welcome.
  converted_parts = self._genai_part_converter(part)
/usr/local/lib/python3.12/dist-packages/google/adk/a2a/converters/event_converter.py:239: UserWarning: [EXPERIMENTAL] convert_a2a_message_to_event: ADK Implementation for A2A support (A2aAgentExecutor, RemoteA2aAgent and corresponding supporting components etc.) is in experimental mode and is subjected to breaking changes. A2A protocol and SDK arethemselves not experimental. Once it's stable enough the experimental mode will be removed. Your feedback is welcome.
  return convert_a

CreditsAnalysisCoordinator > The external course cannot be transferred due to grade requirements (only A or B are accepted).



When here the External(internal to the external university system) course does not exist.

In [ ]:
response = await runner.run_debug("Initialize Transfer credit for student_id 'S1005', internal_crs_num 'PH-348', and external_crs_num 'MA-000'")


 ### Continue session: debug_session_id

User > Initialize Transfer credit for student_id 'S1005', internal_crs_num 'PH-348', and external_crs_num 'MA-000'
CreditsAnalysisCoordinator > Parameters extracted successfully.



/usr/local/lib/python3.12/dist-packages/google/adk/agents/remote_a2a_agent.py:389: UserWarning: [EXPERIMENTAL] convert_genai_part_to_a2a_part: ADK Implementation for A2A support (A2aAgentExecutor, RemoteA2aAgent and corresponding supporting components etc.) is in experimental mode and is subjected to breaking changes. A2A protocol and SDK arethemselves not experimental. Once it's stable enough the experimental mode will be removed. Your feedback is welcome.
  converted_parts = self._genai_part_converter(part)
/usr/local/lib/python3.12/dist-packages/google/adk/a2a/converters/event_converter.py:239: UserWarning: [EXPERIMENTAL] convert_a2a_message_to_event: ADK Implementation for A2A support (A2aAgentExecutor, RemoteA2aAgent and corresponding supporting components etc.) is in experimental mode and is subjected to breaking changes. A2A protocol and SDK arethemselves not experimental. Once it's stable enough the experimental mode will be removed. Your feedback is welcome.
  return convert_a

CreditsAnalysisCoordinator > No internal course information found for the provided details.



Another **grade requirement not met** test below.

In [ ]:
response = await runner.run_debug("Initialize Transfer credit for student_id 'S1005', internal_crs_num 'MA-000', and external_crs_num 'PH-350'")


 ### Continue session: debug_session_id

User > Initialize Transfer credit for student_id 'S1005', internal_crs_num 'MA-000', and external_crs_num 'PH-350'
CreditsAnalysisCoordinator > Parameters extracted successfully.

CreditsAnalysisCoordinator > The external course cannot be transferred due to grade requirements (only A or B are accepted).



Another Poor match.**Similarity score 0.13.**

In [ ]:
response = await runner.run_debug("Initialize Transfer credit for student_id 'S1001', internal_crs_num 'MA-413', and external_crs_num 'CS-450'")


 ### Continue session: debug_session_id

User > Initialize Transfer credit for student_id 'S1001', internal_crs_num 'MA-413', and external_crs_num 'CS-450'
CreditsAnalysisCoordinator > Parameters extracted successfully.

CreditsAnalysisCoordinator > External course information retrieved and grade is acceptable.

[OBSERVABILITY] get_internal_student_course_info called with student_id=S1001, crs_num=MA-413, df_cus_shape=(100, 9)
[OBSERVABILITY] get_internal_student_course_info returned result: [{'student_id': 'S1001', 'crs_major': 'Mathematics', 'crs_num': 'MA-413', 'crs_title': 'Complex Analysis', 'crs_description': 'Theory and application of functions of a complex variable.', 'crs_syllabus': "Apply the Cauchy-Riemann equations to determine complex differentiability. Compute complex integrals using Cauchy's Integral Formula and the Residue Theorem. Represent functions using Taylor and Laurent series expansions. Utilize conformal mappings to solve boundary value problems.", 'num_of_cred

###Summary: Credit Transfer Analysis System

In this project I developed a multi-agent **Credit Transfer Analysis System** designed to automate and streamline the evaluation of external course credits for transfer to an internal university. The primary goal was to provide an efficient and robust mechanism for assessing course equivalence, ensuring academic integrity, and reducing administrative overhead. The system addresses inherent complexities in credit transfer, such as syllabus comparison, grade verification, handling varied institutional systems, and reducing manual effort prone to delays and errors.

### Multi-Agent System Architecture:
The system employs a **multi-agent system architecture**, leveraging the **Google Agent Development Kit (ADK)** and **Agent-to-Agent (A2A) communication**. This modular design enhances robustness, scalability, and efficiency.

Key components include:
*   **CreditsAnalysisCoordinator (Root Agent):** The central orchestrator, managing the entire workflow from query parsing to final analysis, delegating tasks to sub-agents and tools.
*   **query_extraction_agent:** A specialized LLM agent extracting `student_id`, `internal_crs_num`, and `external_crs_num` from user queries.
*   **external_student_course_info_agent (Remote A2A Agent):** An independent agent running remotely, providing external university course information from `df_lus`. It exposes its functionality via the A2A protocol.
*   **query_extraction_agent(external):** A specialized LLM agent extracting `student_id`, and `external_crs_num` from user queries.
*   **remote_external_student_course_info_agent (Local Proxy for A2A):** A client-side proxy enabling the `CreditsAnalysisCoordinator` to interact with the remote agent seamlessly.
*   **Tools (get_internal_student_course_info, compare_syllabi, get_student_info(external)):** Python functions for retrieving internal course data from `df_cus`(internal databse) and 'df_Lus'(external database) and comparing course syllabi using TF-IDF and cosine similarity, respectively.

### Workflow and Observability:
The `CreditsAnalysisCoordinator` orchestrates a sequential workflow: extracting parameters, retrieving and validating external course data, retrieving internal course data, comparing syllabi, and finally generating a comprehensive analysis report. Workflow state and context are maintained by passing intermediate results between agents. **Observability** is integrated throughout, using `[OBSERVABILITY]` prefixed `print` statements within tool functions and `Output` statements in agent instructions, providing real-time insights into execution flow, parameters, and results for effective debugging.

### Key Findings and Achievements:
*   Successful automation of query parsing and distributed information retrieval from disparate data sources.
*   Robust inter-agent communication facilitated by the A2A framework.
*   Intelligent syllabus comparison providing both quantitative scores and qualitative assessments.
*   A flexible, orchestrated workflow capable of handling various scenarios, including missing information, grade requirements, and data unavailability.

### Future Enhancements:
To evolve the system further, the following enhancements are critical:
1.  **Robust Remote Agent Deployment (A2A on Google Cloud):** Deploy the `external_student_course_info_agent` to a more robust, always-on environment within Google Cloud, exposing it reliably via A2A. The current `uvicorn` server is not suitable for continuous operation and was a primary limitation for extensive evaluation and long-term availability.
2.  **End-to-End Authentication:** Implement comprehensive authentication mechanisms at every step of the enterprise architecture. This is crucial for securing sensitive student data and ensuring authorized access to all components and services.
3.  **Cloud-Hosted External Database:** Migrate the external course database (currently simulated by `df_lus` saved to CSV) to a persistent and scalable cloud-hosted database solution within Google Cloud (e.g., Cloud SQL or Firestore). This would enhance data integrity, availability, and management.
4.  **Extensive Evaluation Framework:** Develop and integrate a comprehensive evaluation framework to systematically test the system's performance, accuracy, and reliability across a wide range of real-world scenarios and datasets. This would include quantitative metrics for extraction accuracy, syllabus comparison efficacy, and overall workflow success rates.